# Notebook 04: Golden Dataset Construction for RAGAS

Builds a 100-question RAGAS-ready golden dataset from MedQA + 18 medical textbooks, using **GPT-4** as evidence selector / reference-answer generator / validator.

**Inputs (must exist on Drive from Notebook 01):**
- `data/processed/medqa_dev.parquet`
- `data/processed/textbook_corpus.json`

**Output:**
- `data/processed/medqa_ragas_golden.jsonl` — accepted golden samples for downstream RAGAS evaluation.

**Pipeline (each stage checkpointed to disk):**
1. Stratified sample 100 dev questions (40 Step 1 + 40 Step 2&3 + 20 long-vignette).
2. Build retrieval index — same config as Notebook 02a (200-token semantic chunks, all-MiniLM-L6-v2, ChromaDB + BM25).
3. Hybrid retrieve top-10 candidate evidence chunks per question.
4. GPT-4 evidence-selection pass — pick the chunks that actually support the answer.
5. GPT-4 reference-answer pass — write `reference_answer` + `reference_explanation` + `hallucination_check_points`.
6. GPT-4 validation pass — score faithfulness / relevance / explanation quality.
7. Automated audit — mechanical checks (answer match, evidence keyword coverage).
8. Save final JSONL filtered to `quality_status == 'accepted'`.

**Cost estimate:** ~300 GPT-4 calls (3 passes × 100 questions). Run on 5 first to verify, then full 100.

## 0. Colab setup — mount Drive & install packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

COLAB_PROJECT_ROOT = '/content/drive/MyDrive/MedQA-Thesis'

import subprocess, sys
packages = [
    'langchain==0.3.25', 'langchain-text-splitters==0.3.11',
    'sentence-transformers', 'chromadb', 'rank-bm25',
    'openai',  # OpenAI Python SDK for GPT-4
    'pandas', 'numpy', 'tqdm', 'pyarrow',
]
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q'] + packages,
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

import os
processed = os.path.join(COLAB_PROJECT_ROOT, 'data', 'processed')
for fname in ['medqa_dev.parquet', 'textbook_corpus.json']:
    if not os.path.exists(os.path.join(processed, fname)):
        raise FileNotFoundError(f'Missing: {processed}/{fname} — run Notebook 01 first.')
print('Setup complete.')

## 1. Imports, config, API key

In [ ]:
import json, re, time, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from rank_bm25 import BM25Okapi

from openai import OpenAI

warnings.filterwarnings('ignore')

# --- Paths ---
PROJECT_ROOT   = Path(COLAB_PROJECT_ROOT)
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_INDICES   = PROJECT_ROOT / 'data' / 'indices'
for d in [DATA_PROCESSED, DATA_INDICES]:
    d.mkdir(parents=True, exist_ok=True)

# Cached artifacts — built once in §3, reused on every subsequent run / notebook
CHUNKS_PATH     = DATA_INDICES / 'chunks.parquet'
EMBEDDINGS_PATH = DATA_INDICES / 'embeddings.npy'
CHROMA_DIR      = DATA_INDICES / 'chroma_textbooks'

# --- Sampling ---
N_GOLDEN              = 100
N_STEP1               = 40
N_STEP2_3             = 40
N_LONG_VIGNETTE       = 20
LONG_VIGNETTE_WORDS   = 200
RANDOM_SEED           = 42

# --- Retrieval (matches Notebook 02a) ---
CHUNK_MAX_TOKENS     = 200
CHUNK_OVERLAP        = 20
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'
TOP_K_CANDIDATES     = 10
RRF_K                = 60

# --- OpenAI / GPT-4 ---
OPENAI_API_KEY = ''  # paste here, or leave empty to read from Colab Secrets
OPENAI_MODEL   = 'gpt-4o'  # other valid choices: 'gpt-4-turbo', 'gpt-4.1', 'gpt-4o-2024-11-20'
OPENAI_RPS_DELAY = 0.5  # seconds between calls (be polite)

if not OPENAI_API_KEY:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    raise RuntimeError('Set OPENAI_API_KEY inline or in Colab Secrets.')

openai_client = OpenAI(api_key=OPENAI_API_KEY)
print(f'OpenAI client ready | model={OPENAI_MODEL}')

## 2. Stratified sample of 100 dev questions

Sampling order: **long-vignette first** (rare bucket), then top up Step 1 and Step 2&3 from the remainder so total = 100 unique.

In [ ]:
df_dev = pd.read_parquet(DATA_PROCESSED / 'medqa_dev.parquet')
df_dev['n_words'] = df_dev['question'].str.split().str.len()
print(f'Dev split: {len(df_dev)} questions')

rng_kwargs = dict(random_state=RANDOM_SEED)

# 1) Long-vignette bucket — sample 20 across both steps
long_pool = df_dev[df_dev['n_words'] > LONG_VIGNETTE_WORDS]
long_sample = long_pool.sample(n=min(N_LONG_VIGNETTE, len(long_pool)), **rng_kwargs)

# 2) Step 1 — sample 40 from remainder
remaining = df_dev.drop(long_sample.index)
step1_pool = remaining[remaining['meta_info'] == 'step1']
step1_sample = step1_pool.sample(n=N_STEP1, **rng_kwargs)

# 3) Step 2&3 — sample 40 from remainder
remaining = remaining.drop(step1_sample.index)
step23_pool = remaining[remaining['meta_info'].isin(['step2&3', 'step2_3', 'step2'])]
if len(step23_pool) < N_STEP2_3:
    # fallback: anything not step1
    step23_pool = remaining[remaining['meta_info'] != 'step1']
step23_sample = step23_pool.sample(n=N_STEP2_3, **rng_kwargs)

df_golden_seed = pd.concat([long_sample, step1_sample, step23_sample]).reset_index(drop=True)
df_golden_seed['golden_id'] = [f'medqa_gold_{i:03d}' for i in range(1, len(df_golden_seed) + 1)]

assert len(df_golden_seed) == N_GOLDEN, f'Expected {N_GOLDEN} questions, got {len(df_golden_seed)}'
assert df_golden_seed['golden_id'].is_unique

df_golden_seed.to_parquet(DATA_PROCESSED / 'golden_seed_100.parquet', index=False)
print('\nStratification breakdown:')
print(df_golden_seed.groupby(['meta_info']).size().to_string())
print(f"\nLong-vignette (>{LONG_VIGNETTE_WORDS} words): {(df_golden_seed['n_words'] > LONG_VIGNETTE_WORDS).sum()}")
print(f"Saved → {DATA_PROCESSED / 'golden_seed_100.parquet'}")

## 3. Build retrieval index (cached)

Same chunking + embedding config as Notebook 02a so retrieval here is consistent with downstream RAG experiments.

**Idempotent**: on first run, builds chunks + embeddings + ChromaDB and saves them to `data/indices/`. On every subsequent run (this notebook or any other), it loads the cached artifacts instead — turning a ~10 min build into a ~30 s load.

To force a rebuild, delete the files in `data/indices/`.

In [ ]:
def semantic_passage_chunk(text, max_tokens=200, overlap=20):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_tokens * 4, chunk_overlap=overlap,
        separators=['\n\n\n', '\n\n', '\n', '. ', '; ', ', ', ' '],
        length_function=lambda x: len(x.split()), is_separator_regex=False,
    )
    chunks = splitter.split_text(text)
    return [c.strip() for c in chunks if len(c.split()) >= 10]

# --- Load or build chunks + embeddings ---
if CHUNKS_PATH.exists() and EMBEDDINGS_PATH.exists():
    print(f'Loading cached chunks + embeddings from {DATA_INDICES} ...')
    chunks_df = pd.read_parquet(CHUNKS_PATH)
    all_chunks = chunks_df['text'].tolist()
    chunk_metadata = chunks_df.to_dict(orient='records')
    embeddings = np.load(EMBEDDINGS_PATH)
    embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)  # still needed for encoding query strings
    print(f'Loaded {len(all_chunks):,} chunks | embeddings: {embeddings.shape}')
else:
    print('No cached index found — building from textbook_corpus.json ...')
    with open(DATA_PROCESSED / 'textbook_corpus.json') as f:
        textbook_corpus = json.load(f)

    all_chunks, chunk_metadata = [], []
    for book in tqdm(textbook_corpus, desc='Chunking'):
        for i, ct in enumerate(semantic_passage_chunk(book['text'], CHUNK_MAX_TOKENS, CHUNK_OVERLAP)):
            all_chunks.append(ct)
            chunk_metadata.append({
                'book_name': book['book_name'],
                'chunk_id': f"{book['book_name']}_chunk_{i}",
                'text': ct,
            })
    print(f'Total chunks: {len(all_chunks):,}')

    embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    embeddings = embed_model.encode(all_chunks, show_progress_bar=True, batch_size=256, normalize_embeddings=True)
    embeddings = np.array(embeddings, dtype=np.float32)
    print(f'Embeddings: {embeddings.shape}')

    # Persist for reuse
    pd.DataFrame(chunk_metadata).to_parquet(CHUNKS_PATH, index=False)
    np.save(EMBEDDINGS_PATH, embeddings)
    print(f'Saved chunks      → {CHUNKS_PATH}')
    print(f'Saved embeddings  → {EMBEDDINGS_PATH}')

# --- ChromaDB (persistent on Drive) ---
chroma = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma.get_or_create_collection(name='medqa_textbooks', metadata={'hnsw:space': 'cosine'})

if collection.count() != len(all_chunks):
    print(f'ChromaDB count ({collection.count():,}) != chunks ({len(all_chunks):,}) — rebuilding ...')
    try:
        chroma.delete_collection('medqa_textbooks')
    except Exception:
        pass
    collection = chroma.create_collection(name='medqa_textbooks', metadata={'hnsw:space': 'cosine'})
    BATCH = 5000
    for start in tqdm(range(0, len(all_chunks), BATCH), desc='Indexing ChromaDB'):
        end = min(start + BATCH, len(all_chunks))
        collection.add(
            ids=[str(i) for i in range(start, end)],
            embeddings=embeddings[start:end].tolist(),
            documents=all_chunks[start:end],
            metadatas=[{'book_name': chunk_metadata[i]['book_name'], 'chunk_id': chunk_metadata[i]['chunk_id']} for i in range(start, end)],
        )
    print(f'ChromaDB persisted to {CHROMA_DIR}')
else:
    print(f'ChromaDB already populated ({collection.count():,} vectors) — skipping rebuild')

# --- BM25 (always rebuilt — cheap, ~5 sec) ---
tokenized = [c.lower().split() for c in all_chunks]
bm25 = BM25Okapi(tokenized)
print('BM25 index ready')

## 4. Retrieve top-10 candidate evidence per question

Hybrid retrieval (dense + BM25, RRF-fused). Search query = `question + answer + metamap_phrases` so the index is biased toward chunks that mention the gold answer concept (this is allowed — we are constructing the golden set, not evaluating a system).

In [ ]:
def dense_retrieve(query_emb, k):
    res = collection.query(query_embeddings=[query_emb.tolist()], n_results=k)
    return [int(i) for i in res['ids'][0]], res['distances'][0]

def bm25_retrieve(query_text, k):
    scores = bm25.get_scores(query_text.lower().split())
    idx = np.argsort(scores)[::-1][:k]
    return idx.tolist(), scores[idx].tolist()

def rrf_fuse(*ranking_lists, rrf_k=60):
    fused = {}
    for ranking in ranking_lists:
        for rank, (idx, _) in enumerate(ranking):
            fused[idx] = fused.get(idx, 0) + 1.0 / (rrf_k + rank + 1)
    return sorted(fused.items(), key=lambda x: x[1], reverse=True)

def coerce_to_list(x):
    """metamap_phrases is stored as a numpy array in parquet — coerce to plain list."""
    if x is None:
        return []
    if isinstance(x, (list, tuple)):
        return list(x)
    if isinstance(x, np.ndarray):
        return x.tolist()
    return [x]

def build_search_query(row):
    parts = [row['question'], str(row.get('answer', ''))]
    mm = coerce_to_list(row.get('metamap_phrases'))
    if mm:
        parts.append(' '.join(str(m) for m in mm))
    return ' '.join(parts)

candidates_path = DATA_PROCESSED / 'golden_candidates.jsonl'
with open(candidates_path, 'w') as f:
    for _, row in tqdm(df_golden_seed.iterrows(), total=len(df_golden_seed), desc='Retrieving'):
        query = build_search_query(row)
        q_emb = embed_model.encode(query, normalize_embeddings=True)
        fetch_k = TOP_K_CANDIDATES * 3
        d_idx, d_scores = dense_retrieve(q_emb, fetch_k)
        s_idx, s_scores = bm25_retrieve(query, fetch_k)
        fused = rrf_fuse(list(zip(d_idx, d_scores)), list(zip(s_idx, s_scores)), rrf_k=RRF_K)[:TOP_K_CANDIDATES]
        candidates = [
            {
                'rank': r + 1,
                'chunk_id': chunk_metadata[i]['chunk_id'],
                'book_name': chunk_metadata[i]['book_name'],
                'text': all_chunks[i],
                'rrf_score': float(score),
                'global_idx': int(i),
            }
            for r, (i, score) in enumerate(fused)
        ]
        record = {
            'golden_id': row['golden_id'],
            'question_id': row.get('id', row['golden_id']),
            'question': row['question'],
            'options': {'A': row['option_a'], 'B': row['option_b'], 'C': row['option_c'], 'D': row['option_d']},
            'correct_answer': row['answer'],
            'answer_idx': row['answer_idx'],
            'meta_info': row['meta_info'],
            'metamap_phrases': coerce_to_list(row.get('metamap_phrases')),
            'n_words': int(row['n_words']),
            'candidates': candidates,
        }
        f.write(json.dumps(record, default=str) + '\n')
print(f'Saved → {candidates_path}')

# Quick spot-check
with open(candidates_path) as f:
    sample = json.loads(f.readline())
print(f"\nSpot-check {sample['golden_id']}: top-3 books =",
      [c['book_name'] for c in sample['candidates'][:3]])

## 5. OpenAI / GPT-4 helper

Wraps the OpenAI client with JSON-mode output (`response_format={"type": "json_object"}`), retry on rate-limit / transient errors, and a small per-call delay.

In [ ]:
def gpt_json(prompt: str, temperature: float = 0.0, max_retries: int = 4) -> dict:
    """Call GPT-4 in JSON mode and parse the response. Retries on transient errors.

    Note: OpenAI JSON mode requires the prompt to mention 'JSON' — all our prompts do.
    """
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = openai_client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {'role': 'system', 'content': 'You return strict JSON only — no prose, no markdown fences.'},
                    {'role': 'user', 'content': prompt},
                ],
                temperature=temperature,
                response_format={'type': 'json_object'},
            )
            text = resp.choices[0].message.content
            time.sleep(OPENAI_RPS_DELAY)
            return json.loads(text)
        except json.JSONDecodeError as e:
            last_err = e
            time.sleep(2 ** attempt)
        except Exception as e:
            last_err = e
            msg = str(e).lower()
            if 'rate' in msg or 'quota' in msg or '429' in msg or '503' in msg or 'timeout' in msg:
                time.sleep(5 * (2 ** attempt))
            else:
                time.sleep(2 ** attempt)
    raise RuntimeError(f'GPT-4 call failed after {max_retries} retries: {last_err}')

# Sanity check — one call
_test = gpt_json('Return JSON: {"ok": true, "model": "<model name string>"}.')
print(f'GPT-4 OK: {_test}')

## 6. Pass 1 — Evidence selection

GPT-4 reviews the 10 candidate chunks and picks which ones actually support the correct answer. Output: `selected_chunks` (with support level) + `best_gold_context` + `evidence_keywords`.

In [ ]:
EVIDENCE_PROMPT = '''You are a medical evidence verification assistant.

You will receive a USMLE-style multiple-choice question, the verified correct answer, and 10 candidate textbook passages retrieved from a medical knowledge base.

Task:
- Identify which passages directly support the correct answer.
- Reject passages that are irrelevant, too general, or insufficient on their own.
- Return STRICT JSON only — no prose, no markdown.

Question:
{question}

Options:
{options}

Correct answer ({answer_idx}): {correct_answer}

Candidate passages:
{candidates}

Return JSON with this exact schema:
{{
  "selected_chunks": [
    {{"chunk_id": "<from candidate>", "book_name": "<from candidate>", "support_level": "strong|moderate|weak", "reason": "<1 sentence>"}}
  ],
  "best_gold_context": "<concatenated text of the 1-3 strongest selected passages, verbatim>",
  "evidence_keywords": ["<medical term>", ...],
  "is_evidence_sufficient": true|false,
  "review_note": "<short note if anything is concerning>"
}}'''

def format_options(opts: dict) -> str:
    return '\n'.join(f'{k}) {v}' for k, v in opts.items())

def format_candidates(cands: list) -> str:
    blocks = []
    for c in cands:
        blocks.append(f"[{c['rank']}] chunk_id={c['chunk_id']} | book={c['book_name']}\n{c['text']}")
    return '\n\n'.join(blocks)

evidence_path = DATA_PROCESSED / 'golden_evidence_selected.jsonl'
with open(candidates_path) as fin, open(evidence_path, 'w') as fout:
    for line in tqdm(list(fin), desc='Evidence selection'):
        rec = json.loads(line)
        prompt = EVIDENCE_PROMPT.format(
            question=rec['question'],
            options=format_options(rec['options']),
            answer_idx=rec['answer_idx'],
            correct_answer=rec['correct_answer'],
            candidates=format_candidates(rec['candidates']),
        )
        try:
            result = gpt_json(prompt, temperature=0.0)
            rec['evidence_selection'] = result
            rec['evidence_selection_status'] = 'ok'
        except Exception as e:
            rec['evidence_selection'] = None
            rec['evidence_selection_status'] = f'error: {e}'
        fout.write(json.dumps(rec, default=str) + '\n')

# Summary
errs = sum(1 for line in open(evidence_path) if json.loads(line)['evidence_selection_status'] != 'ok')
print(f'Saved → {evidence_path} | failed: {errs}/100')

## 7. Pass 2 — Reference answer + explanation

Given the gold context, GPT-4 writes a clean reference answer, an evidence-grounded explanation, and explicit `hallucination_check_points` (claims a faithful generated answer must support).

In [ ]:
REFERENCE_PROMPT = '''You are creating a reference answer for a medical question-answering evaluation dataset.

Use ONLY the provided question, correct answer, and gold textbook evidence. Do not introduce medical claims that the evidence does not support. Write in clear academic English.

Question:
{question}

Options:
{options}

Correct answer ({answer_idx}): {correct_answer}

Gold evidence:
{gold_context}

Return STRICT JSON only — no prose, no markdown — with this schema:
{{
  "reference_answer": "<single-sentence answer that names the correct option>",
  "reference_explanation": "<3-6 sentence explanation grounded in the evidence>",
  "why_other_options_are_less_suitable": "<1-3 sentences>",
  "hallucination_check_points": ["<a claim a faithful answer must support>", ...],
  "question_type": "diagnosis|treatment|mechanism|management|other",
  "requires_multihop": "yes|no"
}}'''

references_path = DATA_PROCESSED / 'golden_with_references.jsonl'
with open(evidence_path) as fin, open(references_path, 'w') as fout:
    for line in tqdm(list(fin), desc='Reference answers'):
        rec = json.loads(line)
        ev = rec.get('evidence_selection') or {}
        gold_context = ev.get('best_gold_context', '') if rec['evidence_selection_status'] == 'ok' else ''
        if not gold_context:
            rec['reference'] = None
            rec['reference_status'] = 'skipped: no gold context'
            fout.write(json.dumps(rec, default=str) + '\n')
            continue
        prompt = REFERENCE_PROMPT.format(
            question=rec['question'],
            options=format_options(rec['options']),
            answer_idx=rec['answer_idx'],
            correct_answer=rec['correct_answer'],
            gold_context=gold_context,
        )
        try:
            result = gpt_json(prompt, temperature=0.2)
            rec['reference'] = result
            rec['reference_status'] = 'ok'
        except Exception as e:
            rec['reference'] = None
            rec['reference_status'] = f'error: {e}'
        fout.write(json.dumps(rec, default=str) + '\n')

errs = sum(1 for line in open(references_path) if json.loads(line)['reference_status'] != 'ok')
print(f'Saved → {references_path} | failed/skipped: {errs}/100')

## 8. Pass 3 — Validation

GPT-4 scores each constructed sample (evidence relevance, faithfulness, explanation quality, hallucination risk) and assigns a `final_status`. This is automated quality control before the audit pass.

In [ ]:
VALIDATION_PROMPT = '''You are validating a golden dataset sample for medical RAG evaluation.

Check whether:
1. The reference answer matches the original correct answer.
2. The explanation is supported by the gold evidence.
3. The evidence is genuinely relevant to the question.
4. There are no unsupported medical claims.
5. The sample is suitable for RAGAS evaluation.

Question: {question}
Correct answer ({answer_idx}): {correct_answer}
Gold evidence: {gold_context}
Reference answer: {reference_answer}
Reference explanation: {reference_explanation}

Return STRICT JSON only:
{{
  "answer_match": true|false,
  "evidence_relevance_score": 0-5,
  "faithfulness_score": 0-5,
  "explanation_quality_score": 0-5,
  "hallucination_risk": "low|medium|high",
  "final_status": "accepted|needs_review|rejected",
  "reason": "<short>"
}}'''

validated_path = DATA_PROCESSED / 'golden_validated.jsonl'
with open(references_path) as fin, open(validated_path, 'w') as fout:
    for line in tqdm(list(fin), desc='Validation'):
        rec = json.loads(line)
        ref = rec.get('reference') or {}
        ev = rec.get('evidence_selection') or {}
        if rec.get('reference_status') != 'ok':
            rec['validation'] = None
            rec['validation_status'] = 'skipped'
            fout.write(json.dumps(rec, default=str) + '\n')
            continue
        prompt = VALIDATION_PROMPT.format(
            question=rec['question'],
            answer_idx=rec['answer_idx'],
            correct_answer=rec['correct_answer'],
            gold_context=ev.get('best_gold_context', ''),
            reference_answer=ref.get('reference_answer', ''),
            reference_explanation=ref.get('reference_explanation', ''),
        )
        try:
            result = gpt_json(prompt, temperature=0.0)
            rec['validation'] = result
            rec['validation_status'] = 'ok'
        except Exception as e:
            rec['validation'] = None
            rec['validation_status'] = f'error: {e}'
        fout.write(json.dumps(rec, default=str) + '\n')

# Status distribution
statuses = Counter()
for line in open(validated_path):
    rec = json.loads(line)
    v = rec.get('validation') or {}
    statuses[v.get('final_status', rec.get('validation_status', 'missing'))] += 1
print(f'Saved → {validated_path}')
print('Status distribution:', dict(statuses))

## 9. Automated audit

Mechanical, non-LLM checks that don't require a clinician:
- Does `reference_answer` mention the MedQA gold-correct answer string? (catches GPT-4 disagreeing with the label)
- Do the gold-context passages contain ≥1 of the declared `evidence_keywords`? (catches keyword hallucination)
- Are all cited `chunk_id`s real (i.e., from our index)?

Anything that fails → `quality_status = 'needs_review'` regardless of the LLM's own validation verdict.

In [ ]:
valid_chunk_ids = {m['chunk_id'] for m in chunk_metadata}

def audit(rec: dict) -> dict:
    issues = []
    ev = rec.get('evidence_selection') or {}
    ref = rec.get('reference') or {}
    val = rec.get('validation') or {}

    if rec.get('evidence_selection_status') != 'ok':
        issues.append('evidence_selection_failed')
    if rec.get('reference_status') != 'ok':
        issues.append('reference_failed')
    if rec.get('validation_status') != 'ok':
        issues.append('validation_failed')

    # Answer-string presence in reference_answer
    ref_text = (ref.get('reference_answer', '') or '').lower()
    if rec.get('correct_answer', '').lower() not in ref_text:
        issues.append('correct_answer_not_in_reference_answer')

    # Evidence keyword coverage
    gold_text = (ev.get('best_gold_context', '') or '').lower()
    keywords = [k.lower() for k in (ev.get('evidence_keywords') or []) if k]
    if keywords and not any(k in gold_text for k in keywords):
        issues.append('no_evidence_keyword_in_gold_context')

    # Chunk ID validity
    cited = [c.get('chunk_id') for c in (ev.get('selected_chunks') or [])]
    bad_ids = [cid for cid in cited if cid and cid not in valid_chunk_ids]
    if bad_ids:
        issues.append(f'invalid_chunk_ids:{len(bad_ids)}')

    # Final quality_status: combine LLM verdict + audit
    llm_status = (val.get('final_status') or '').lower()
    if issues:
        quality_status = 'needs_review'
    elif llm_status == 'accepted':
        quality_status = 'accepted'
    elif llm_status == 'rejected':
        quality_status = 'rejected'
    else:
        quality_status = 'needs_review'

    return {'audit_issues': issues, 'quality_status': quality_status}

audited_path = DATA_PROCESSED / 'golden_audited.jsonl'
with open(validated_path) as fin, open(audited_path, 'w') as fout:
    for line in fin:
        rec = json.loads(line)
        rec.update(audit(rec))
        fout.write(json.dumps(rec, default=str) + '\n')

audit_summary = Counter()
for line in open(audited_path):
    rec = json.loads(line)
    audit_summary[rec['quality_status']] += 1
    for iss in rec['audit_issues']:
        audit_summary[f'issue:{iss}'] += 1
print(f'Saved → {audited_path}')
print('Audit summary:', dict(audit_summary))

## 10. Save final golden dataset

Filter to `quality_status == 'accepted'` and emit a flat record per row in the format the downstream RAGAS notebook will consume.

In [ ]:
final_path = DATA_PROCESSED / 'medqa_ragas_golden.jsonl'
review_path = DATA_PROCESSED / 'medqa_ragas_golden_needs_review.jsonl'

kept = 0
review = 0
with open(audited_path) as fin, open(final_path, 'w') as fout, open(review_path, 'w') as freview:
    for line in fin:
        rec = json.loads(line)
        ev = rec.get('evidence_selection') or {}
        ref = rec.get('reference') or {}
        val = rec.get('validation') or {}
        flat = {
            'golden_id': rec['golden_id'],
            'question_id': rec['question_id'],
            'question': rec['question'],
            'options': rec['options'],
            'correct_answer': rec['correct_answer'],
            'answer_idx': rec['answer_idx'],
            'meta_info': rec['meta_info'],
            'metamap_phrases': rec['metamap_phrases'],
            'gold_context': ev.get('best_gold_context', ''),
            'gold_chunks': ev.get('selected_chunks', []),
            'evidence_keywords': ev.get('evidence_keywords', []),
            'reference_answer': ref.get('reference_answer', ''),
            'reference_explanation': ref.get('reference_explanation', ''),
            'why_other_options_are_less_suitable': ref.get('why_other_options_are_less_suitable', ''),
            'hallucination_check_points': ref.get('hallucination_check_points', []),
            'question_type': ref.get('question_type', ''),
            'requires_multihop': ref.get('requires_multihop', ''),
            'validation_scores': {
                'evidence_relevance_score': val.get('evidence_relevance_score'),
                'faithfulness_score': val.get('faithfulness_score'),
                'explanation_quality_score': val.get('explanation_quality_score'),
                'hallucination_risk': val.get('hallucination_risk'),
            },
            'audit_issues': rec['audit_issues'],
            'quality_status': rec['quality_status'],
        }
        if rec['quality_status'] == 'accepted':
            fout.write(json.dumps(flat, default=str) + '\n')
            kept += 1
        elif rec['quality_status'] == 'needs_review':
            freview.write(json.dumps(flat, default=str) + '\n')
            review += 1

print(f'Final golden dataset: {kept} accepted → {final_path}')
print(f'Needs human review:  {review} → {review_path}')
print('\nNext: open the needs-review file, fix or drop rows manually, then merge into the final.')

## Done

**Artifacts on Drive:**

`data/indices/` — built once, reused by every notebook
- `chunks.parquet` — ~70k textbook chunks with `book_name` + `chunk_id` + `text`.
- `embeddings.npy` — 70k × 384 float32 sentence-transformer embeddings.
- `chroma_textbooks/` — persistent ChromaDB collection.

`data/processed/` — golden-set construction outputs
- `golden_seed_100.parquet` — the 100 sampled questions (raw seed).
- `golden_candidates.jsonl` — top-10 retrieved candidates per question.
- `golden_evidence_selected.jsonl` — GPT-4 Pass 1 output.
- `golden_with_references.jsonl` — GPT-4 Pass 2 output.
- `golden_validated.jsonl` — GPT-4 Pass 3 output.
- `golden_audited.jsonl` — full record + audit verdicts.
- `medqa_ragas_golden.jsonl` — **final golden dataset** (accepted only).
- `medqa_ragas_golden_needs_review.jsonl` — rows for manual review.

**Each LLM stage is checkpointed** — if Pass 2 or 3 fails, you can re-run just that cell without re-doing prior passes.

**Recommended manual step:** open `medqa_ragas_golden_needs_review.jsonl`, look at `audit_issues` and `validation_scores`, and either fix the row (e.g., regenerate Pass 2 with a different prompt) or drop it. Don't merge anything you wouldn't defend in a viva.